In [1]:
import sys
!{sys.executable} -m pip install -q pandas==2.2.3 rapidfuzz==3.9.7 tqdm==4.66.5


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import re
import time
import unicodedata

import pandas as pd
from rapidfuzz import fuzz
from tqdm.auto import tqdm


C:\Users\Giovanny\anaconda3\envs\fluidgpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CWD = Path.cwd().resolve()
BASE_DIR = CWD if CWD.name == "notebooks" else CWD / "notebooks"
OUTPUT_DIR = BASE_DIR / "output"
CACHE_DIR = BASE_DIR / "cache"
RAW_DIR = BASE_DIR / "raw"
REPORTS_DIR = BASE_DIR / "reports"

for directory in [BASE_DIR, OUTPUT_DIR, CACHE_DIR, RAW_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


BASE_DIR: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks
OUTPUT_DIR: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output


In [4]:
INPUT_COLUMNS = [
    "nombre_persona_input", "estado_input", "municipio_input", "puesto_input", "dependencia_input",
    "periodo_inicio_input", "periodo_fin_input", "partido_input", "rfc_input", "curp_input",
    "empresa_relacionada_input", "alias_input", "notas_input",
]

NORMALIZED_COLUMNS = [
    "seed_id", "nombre_persona_input", "normalized_name_input", "token_sort_name", "name_tokens",
    "estado_input", "municipio_input", "puesto_input", "dependencia_input", "periodo_inicio_input",
    "periodo_fin_input", "partido_input", "rfc_input", "curp_input", "empresa_relacionada_input",
    "alias_input", "has_only_name", "input_quality_score", "created_at",
]

PARTICLES = {"de", "del", "la", "las", "los", "el", "y"}
CONTEXT_COLUMNS = [
    "estado_input", "municipio_input", "puesto_input", "dependencia_input", "periodo_inicio_input",
    "periodo_fin_input", "partido_input", "rfc_input", "curp_input", "empresa_relacionada_input", "alias_input",
]


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def remove_accents(value):
    if pd.isna(value):
        return ""
    return "".join(ch for ch in unicodedata.normalize("NFKD", str(value)) if not unicodedata.combining(ch))


def normalize_text(value):
    text = remove_accents(value).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_name(value, remove_particles=False):
    tokens = normalize_text(value).split()
    if remove_particles:
        tokens = [token for token in tokens if token not in PARTICLES]
    return " ".join(tokens)


def stable_hash(value):
    return hashlib.sha256(normalize_name(value).encode("utf-8")).hexdigest()[:16]


def score_name_match(left, right):
    left_norm = normalize_name(left)
    right_norm = normalize_name(right)
    if not left_norm or not right_norm:
        return 0.0
    left_tokens = [token for token in left_norm.split() if token not in PARTICLES]
    right_tokens = [token for token in right_norm.split() if token not in PARTICLES]
    if not left_tokens or not right_tokens:
        return 0.0
    left_set = set(left_tokens)
    right_set = set(right_tokens)
    overlap = len(left_set & right_set)
    if overlap == 0:
        return 0.0
    smaller = min(len(left_set), len(right_set))
    larger = max(len(left_set), len(right_set))
    overlap_ratio = overlap / smaller if smaller else 0.0
    scores = [
        fuzz.WRatio(left_norm, right_norm),
        fuzz.token_sort_ratio(left_norm, right_norm),
        fuzz.ratio(left_norm, right_norm),
    ]
    if smaller >= 2 and overlap_ratio >= 0.67:
        scores.append(fuzz.token_set_ratio(left_norm, right_norm))
    raw_score = float(max(scores))
    if smaller == 1 and larger >= 3:
        raw_score = min(raw_score, 59.0)
    elif smaller == 2 and larger >= 4:
        raw_score = min(raw_score, 74.0)
    if overlap_ratio < 0.50:
        raw_score = min(raw_score, 59.0)
    elif overlap_ratio < 0.67:
        raw_score = min(raw_score, 74.0)
    return round(raw_score, 2)


def classify_match(score):
    score = float(score or 0)
    if score >= 95:
        return "exact_or_very_strong_name"
    if score >= 88:
        return "strong_candidate"
    if score >= 75:
        return "medium_candidate"
    if score >= 60:
        return "weak_candidate"
    return "discard"


def has_only_name(row):
    return not any(str(row.get(column, "")).strip() for column in CONTEXT_COLUMNS)


def input_quality_score(row):
    name_tokens = normalize_name(row.get("nombre_persona_input", "")).split()
    context_count = sum(bool(str(row.get(column, "")).strip()) for column in CONTEXT_COLUMNS)
    score = 0.25 + min(len(name_tokens), 4) * 0.08 + min(context_count, 6) * 0.07
    if len(name_tokens) <= 2:
        score -= 0.15
    if row.get("rfc_input") or row.get("curp_input"):
        score += 0.15
    return round(max(0.0, min(score, 1.0)), 2)


def normalize_input_dataframe(df):
    rows = []
    created_at = now_utc()
    for _, row in df.fillna("").iterrows():
        name = str(row.get("nombre_persona_input", "")).strip()
        normalized = normalize_name(name)
        tokens = normalize_name(name, remove_particles=True).split()
        seed_material = "|".join([
            normalized,
            normalize_text(row.get("estado_input", "")),
            normalize_text(row.get("dependencia_input", "")),
            normalize_text(row.get("rfc_input", "")),
            normalize_text(row.get("curp_input", "")),
        ])
        rows.append({
            "seed_id": hashlib.sha256(seed_material.encode("utf-8")).hexdigest()[:16],
            "nombre_persona_input": name,
            "normalized_name_input": normalized,
            "token_sort_name": " ".join(sorted(tokens)),
            "name_tokens": "|".join(tokens),
            "estado_input": row.get("estado_input", ""),
            "municipio_input": row.get("municipio_input", ""),
            "puesto_input": row.get("puesto_input", ""),
            "dependencia_input": row.get("dependencia_input", ""),
            "periodo_inicio_input": row.get("periodo_inicio_input", ""),
            "periodo_fin_input": row.get("periodo_fin_input", ""),
            "partido_input": row.get("partido_input", ""),
            "rfc_input": row.get("rfc_input", ""),
            "curp_input": row.get("curp_input", ""),
            "empresa_relacionada_input": row.get("empresa_relacionada_input", ""),
            "alias_input": row.get("alias_input", ""),
            "has_only_name": bool(has_only_name(row)),
            "input_quality_score": input_quality_score(row),
            "created_at": created_at,
        })
    return pd.DataFrame(rows, columns=NORMALIZED_COLUMNS)


In [5]:
USER_NAMES = []
# Para reemplazar datos, usa strings o dicts con los campos *_input.
# USER_NAMES = ["Nombre Apellido"]
# USER_NAMES = [{"nombre_persona_input": "Nombre Apellido", "estado_input": "CDMX", "puesto_input": "Cargo"}]

TEST_RECORDS = [
    {"nombre_persona_input": "Claudia Sheinbaum Pardo", "estado_input": "Nacional", "municipio_input": "", "puesto_input": "Presidenta de Mexico", "dependencia_input": "Gobierno de Mexico", "periodo_inicio_input": "2024", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
    {"nombre_persona_input": "Andres Manuel Lopez Obrador", "estado_input": "Nacional", "municipio_input": "", "puesto_input": "Ex presidente de Mexico", "dependencia_input": "Gobierno de Mexico", "periodo_inicio_input": "2018", "periodo_fin_input": "2024", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "AMLO", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
    {"nombre_persona_input": "Marcelo Ebrard Casaubon", "estado_input": "Nacional", "municipio_input": "", "puesto_input": "Servidor publico / figura publica", "dependencia_input": "Gobierno de Mexico", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
    {"nombre_persona_input": "Adan Augusto Lopez Hernandez", "estado_input": "Tabasco", "municipio_input": "", "puesto_input": "Senador / figura publica", "dependencia_input": "Senado de la Republica", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
    {"nombre_persona_input": "Ricardo Monreal Avila", "estado_input": "Zacatecas", "municipio_input": "", "puesto_input": "Legislador / figura publica", "dependencia_input": "Congreso de la Union", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
    {"nombre_persona_input": "Xochitl Galvez Ruiz", "estado_input": "Nacional", "municipio_input": "", "puesto_input": "Senadora / figura publica", "dependencia_input": "Senado de la Republica", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
    {"nombre_persona_input": "Samuel Garcia Sepulveda", "estado_input": "Nuevo Leon", "municipio_input": "", "puesto_input": "Gobernador", "dependencia_input": "Gobierno de Nuevo Leon", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
    {"nombre_persona_input": "Clara Brugada Molina", "estado_input": "Ciudad de Mexico", "municipio_input": "", "puesto_input": "Jefa de Gobierno", "dependencia_input": "Gobierno de la Ciudad de Mexico", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
    {"nombre_persona_input": "Omar Garcia Harfuch", "estado_input": "Nacional", "municipio_input": "", "puesto_input": "Servidor publico / figura publica", "dependencia_input": "Gobierno de Mexico", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
    {"nombre_persona_input": "Luisa Maria Alcalde Lujan", "estado_input": "Nacional", "municipio_input": "", "puesto_input": "Figura publica", "dependencia_input": "Gobierno de Mexico", "periodo_inicio_input": "", "periodo_fin_input": "", "partido_input": "", "rfc_input": "", "curp_input": "", "empresa_relacionada_input": "", "alias_input": "", "notas_input": "Dato de prueba tecnica; no implica senal negativa."},
]

if USER_NAMES:
    prepared = []
    for item in USER_NAMES:
        if isinstance(item, dict):
            row = {column: item.get(column, "") for column in INPUT_COLUMNS}
            if not row["nombre_persona_input"]:
                row["nombre_persona_input"] = item.get("nombre", "")
        else:
            row = {column: "" for column in INPUT_COLUMNS}
            row["nombre_persona_input"] = str(item)
            row["notas_input"] = "Dato ingresado por usuario."
        prepared.append(row)
    input_df = pd.DataFrame(prepared)
else:
    input_df = pd.DataFrame(TEST_RECORDS)

for column in INPUT_COLUMNS:
    if column not in input_df.columns:
        input_df[column] = ""
input_df = input_df[INPUT_COLUMNS].fillna("")
input_path = OUTPUT_DIR / "00_peps_input.csv"
input_df.to_csv(input_path, index=False, encoding="utf-8-sig")
print(f"Input guardado: {input_path}")
display(input_df)


Input guardado: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\00_peps_input.csv


,nombre_persona_input,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,rfc_input,curp_input,empresa_relacionada_input,alias_input,notas_input
0,Claudia Sheinbaum Pardo,Nacional,,Presidenta de Mexico,Gobierno de Mexico,2024,,,,,,,Dato de prueba tecnica; no implica senal negat...
1,Andres Manuel Lopez Obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018,2024,,,,,AMLO,Dato de prueba tecnica; no implica senal negat...
2,Marcelo Ebrard Casaubon,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,Dato de prueba tecnica; no implica senal negat...
3,Adan Augusto Lopez Hernandez,Tabasco,,Senador / figura publica,Senado de la Republica,,,,,,,,Dato de prueba tecnica; no implica senal negat...
4,Ricardo Monreal Avila,Zacatecas,,Legislador / figura publica,Congreso de la Union,,,,,,,,Dato de prueba tecnica; no implica senal negat...
5,Xochitl Galvez Ruiz,Nacional,,Senadora / figura publica,Senado de la Republica,,,,,,,,Dato de prueba tecnica; no implica senal negat...
6,Samuel Garcia Sepulveda,Nuevo Leon,,Gobernador,Gobierno de Nuevo Leon,,,,,,,,Dato de prueba tecnica; no implica senal negat...
7,Clara Brugada Molina,Ciudad de Mexico,,Jefa de Gobierno,Gobierno de la Ciudad de Mexico,,,,,,,,Dato de prueba tecnica; no implica senal negat...
8,Omar Garcia Harfuch,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,Dato de prueba tecnica; no implica senal negat...
9,Luisa Maria Alcalde Lujan,Nacional,,Figura publica,Gobierno de Mexico,,,,,,,,Dato de prueba tecnica; no implica senal negat...


In [6]:
normalized_df = normalize_input_dataframe(input_df)
normalized_path = OUTPUT_DIR / "00_peps_normalized.csv"
normalized_df.to_csv(normalized_path, index=False, encoding="utf-8-sig")
print(f"Normalizado guardado: {normalized_path}")
display(normalized_df)


Normalizado guardado: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\00_peps_normalized.csv


,seed_id,nombre_persona_input,normalized_name_input,token_sort_name,name_tokens,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,rfc_input,curp_input,empresa_relacionada_input,alias_input,has_only_name,input_quality_score,created_at
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,claudia sheinbaum pardo,claudia pardo sheinbaum,claudia|sheinbaum|pardo,Nacional,,Presidenta de Mexico,Gobierno de Mexico,2024,,,,,,,False,0.77,2026-05-07T07:51:29.008342+00:00
1,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,andres lopez manuel obrador,andres|manuel|lopez|obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018,2024,,,,,AMLO,False,0.99,2026-05-07T07:51:29.008342+00:00
2,55c8819cc4622591,Marcelo Ebrard Casaubon,marcelo ebrard casaubon,casaubon ebrard marcelo,marcelo|ebrard|casaubon,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
3,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,adan augusto lopez hernandez,adan augusto hernandez lopez,adan|augusto|lopez|hernandez,Tabasco,,Senador / figura publica,Senado de la Republica,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00
4,2597a81eb6062feb,Ricardo Monreal Avila,ricardo monreal avila,avila monreal ricardo,ricardo|monreal|avila,Zacatecas,,Legislador / figura publica,Congreso de la Union,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
5,39162acdaa914138,Xochitl Galvez Ruiz,xochitl galvez ruiz,galvez ruiz xochitl,xochitl|galvez|ruiz,Nacional,,Senadora / figura publica,Senado de la Republica,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
6,0cafd5ecac30f555,Samuel Garcia Sepulveda,samuel garcia sepulveda,garcia samuel sepulveda,samuel|garcia|sepulveda,Nuevo Leon,,Gobernador,Gobierno de Nuevo Leon,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
7,cba9b1a8ead15ac3,Clara Brugada Molina,clara brugada molina,brugada clara molina,clara|brugada|molina,Ciudad de Mexico,,Jefa de Gobierno,Gobierno de la Ciudad de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
8,69573f0325299b8b,Omar Garcia Harfuch,omar garcia harfuch,garcia harfuch omar,omar|garcia|harfuch,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
9,75c154f2e7b3a23f,Luisa Maria Alcalde Lujan,luisa maria alcalde lujan,alcalde luisa lujan maria,luisa|maria|alcalde|lujan,Nacional,,Figura publica,Gobierno de Mexico,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00


In [7]:
def build_synthetic_input(size, base_df):
    rows = []
    records = base_df.to_dict(orient="records")
    for i in range(size):
        base = dict(records[i % len(records)])
        variant = i // len(records)
        if variant % 4 == 1:
            base["nombre_persona_input"] = base["nombre_persona_input"].upper()
        elif variant % 4 == 2:
            base["nombre_persona_input"] = f"{base['nombre_persona_input']} {variant}"
        elif variant % 4 == 3:
            base["nombre_persona_input"] = f"  {base['nombre_persona_input'].lower()}  "
        rows.append(base)
    return pd.DataFrame(rows, columns=INPUT_COLUMNS)

benchmark_rows = []
for size in [10, 1_000, 10_000]:
    synthetic = build_synthetic_input(size, input_df)
    start = time.perf_counter()
    _ = normalize_input_dataframe(synthetic)
    elapsed = time.perf_counter() - start
    benchmark_rows.append({
        "notebook": "00_input_normalizacion_benchmark",
        "source": "input",
        "step": "normalizacion",
        "duration_seconds": round(elapsed, 6),
        "records_processed": size,
        "records_per_second": round(size / elapsed, 2) if elapsed else 0,
        "status": "ok",
        "error_message": "",
        "notes": "benchmark local de normalizacion",
    })

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_path = OUTPUT_DIR / "00_benchmark_normalizacion.csv"
benchmark_df.to_csv(benchmark_path, index=False, encoding="utf-8-sig")
print(f"Benchmark guardado: {benchmark_path}")
display(benchmark_df)


Benchmark guardado: C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\00_benchmark_normalizacion.csv


,notebook,source,step,duration_seconds,records_processed,records_per_second,status,error_message,notes
0,00_input_normalizacion_benchmark,input,normalizacion,0.002472,10,4044.65,ok,,benchmark local de normalizacion
1,00_input_normalizacion_benchmark,input,normalizacion,0.162225,1000,6164.26,ok,,benchmark local de normalizacion
2,00_input_normalizacion_benchmark,input,normalizacion,1.618812,10000,6177.37,ok,,benchmark local de normalizacion


In [8]:
print("Resumen notebook 00")
print(f"1. Personas procesadas: {len(normalized_df):,}")
print("2. Filas de evidencia generadas: no aplica en notebook 00")
print("3. Personas con hits: no aplica en notebook 00")
print("4. Top 10 filas tabulares:")
display(normalized_df.head(10))
print("5. CSV generados:")
for path in [input_path, normalized_path, benchmark_path]:
    print(f"- {path}")


Resumen notebook 00
1. Personas procesadas: 10
2. Filas de evidencia generadas: no aplica en notebook 00
3. Personas con hits: no aplica en notebook 00
4. Top 10 filas tabulares:


,seed_id,nombre_persona_input,normalized_name_input,token_sort_name,name_tokens,estado_input,municipio_input,puesto_input,dependencia_input,periodo_inicio_input,periodo_fin_input,partido_input,rfc_input,curp_input,empresa_relacionada_input,alias_input,has_only_name,input_quality_score,created_at
0,064afd0cc88b3699,Claudia Sheinbaum Pardo,claudia sheinbaum pardo,claudia pardo sheinbaum,claudia|sheinbaum|pardo,Nacional,,Presidenta de Mexico,Gobierno de Mexico,2024,,,,,,,False,0.77,2026-05-07T07:51:29.008342+00:00
1,910a01d167e16711,Andres Manuel Lopez Obrador,andres manuel lopez obrador,andres lopez manuel obrador,andres|manuel|lopez|obrador,Nacional,,Ex presidente de Mexico,Gobierno de Mexico,2018,2024,,,,,AMLO,False,0.99,2026-05-07T07:51:29.008342+00:00
2,55c8819cc4622591,Marcelo Ebrard Casaubon,marcelo ebrard casaubon,casaubon ebrard marcelo,marcelo|ebrard|casaubon,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
3,15b1d586cc02eb0e,Adan Augusto Lopez Hernandez,adan augusto lopez hernandez,adan augusto hernandez lopez,adan|augusto|lopez|hernandez,Tabasco,,Senador / figura publica,Senado de la Republica,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00
4,2597a81eb6062feb,Ricardo Monreal Avila,ricardo monreal avila,avila monreal ricardo,ricardo|monreal|avila,Zacatecas,,Legislador / figura publica,Congreso de la Union,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
5,39162acdaa914138,Xochitl Galvez Ruiz,xochitl galvez ruiz,galvez ruiz xochitl,xochitl|galvez|ruiz,Nacional,,Senadora / figura publica,Senado de la Republica,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
6,0cafd5ecac30f555,Samuel Garcia Sepulveda,samuel garcia sepulveda,garcia samuel sepulveda,samuel|garcia|sepulveda,Nuevo Leon,,Gobernador,Gobierno de Nuevo Leon,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
7,cba9b1a8ead15ac3,Clara Brugada Molina,clara brugada molina,brugada clara molina,clara|brugada|molina,Ciudad de Mexico,,Jefa de Gobierno,Gobierno de la Ciudad de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
8,69573f0325299b8b,Omar Garcia Harfuch,omar garcia harfuch,garcia harfuch omar,omar|garcia|harfuch,Nacional,,Servidor publico / figura publica,Gobierno de Mexico,,,,,,,,False,0.70,2026-05-07T07:51:29.008342+00:00
9,75c154f2e7b3a23f,Luisa Maria Alcalde Lujan,luisa maria alcalde lujan,alcalde luisa lujan maria,luisa|maria|alcalde|lujan,Nacional,,Figura publica,Gobierno de Mexico,,,,,,,,False,0.78,2026-05-07T07:51:29.008342+00:00


5. CSV generados:
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\00_peps_input.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\00_peps_normalized.csv
- C:\Users\Giovanny\Desktop\web_scraping_peps\notebooks\output\00_benchmark_normalizacion.csv
